# Prepare Calendar Table

Generates a standard date dimension table `adb_databeli.silver.calendar` spanning **1940-01-01 to 2030-12-31** with all key calendar attributes.

In [0]:
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.functions import (
    col, lit, expr, date_format, dayofmonth, dayofweek, dayofyear,
    month, year, quarter, weekofyear, when, last_day, date_add,
    current_timestamp, concat, lpad
)
from pyspark.sql.types import DateType

# Generate date sequence from 1940-01-01 to 2030-12-31
start_date = "1940-01-01"
end_date = "2030-12-31"

calendar_df = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{start_date}'), 
            to_date('{end_date}'), 
            interval 1 day
        )) AS date
    """)
)

print(f"✅ Generated {calendar_df.count():,} days from {start_date} to {end_date}")

In [0]:
calendar_table = (
    calendar_df
    # Date key (YYYYMMDD integer)
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))

    # Year / Month / Day
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))

    # Month names
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("month_short_name", date_format(col("date"), "MMM"))

    # Day of week (1=Sunday .. 7=Saturday in Spark default)
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
    .withColumn("day_short_name", date_format(col("date"), "EEE"))

    # Day of year
    .withColumn("day_of_year", dayofyear(col("date")))

    # Week and Quarter
    .withColumn("week_of_year", weekofyear(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("quarter_name", concat(lit("Q"), quarter(col("date"))))

    # Year-Month / Year-Quarter labels
    .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
    .withColumn("year_quarter", concat(year(col("date")), lit("-Q"), quarter(col("date"))))

    # Weekend flag
    .withColumn("is_weekend", when(dayofweek(col("date")).isin(1, 7), True).otherwise(False))

    # Weekday flag
    .withColumn("is_weekday", when(dayofweek(col("date")).isin(1, 7), False).otherwise(True))

    # Start / End of month
    .withColumn("first_day_of_month", expr("trunc(date, 'MM')"))
    .withColumn("last_day_of_month", last_day(col("date")))
    .withColumn("is_month_start", when(dayofmonth(col("date")) == 1, True).otherwise(False))
    .withColumn("is_month_end", when(col("date") == last_day(col("date")), True).otherwise(False))

    # Start / End of year
    .withColumn("is_year_start", when(
        (month(col("date")) == 1) & (dayofmonth(col("date")) == 1), True
    ).otherwise(False))
    .withColumn("is_year_end", when(
        (month(col("date")) == 12) & (dayofmonth(col("date")) == 31), True
    ).otherwise(False))

    # Leap year
    .withColumn("is_leap_year", when(
        ((year(col("date")) % 4 == 0) & (year(col("date")) % 100 != 0)) |
        (year(col("date")) % 400 == 0), True
    ).otherwise(False))

    # Fiscal year (assuming fiscal year starts April 1)
    .withColumn("fiscal_year", when(
        month(col("date")) >= 4, year(col("date"))
    ).otherwise(year(col("date")) - 1))
    .withColumn("fiscal_quarter", when(
        quarter(col("date")) >= 2, quarter(col("date")) - 1
    ).otherwise(quarter(col("date")) + 3))

    # Audit column
    .withColumn("insert_timestamp", current_timestamp())
)

print(f"✅ Calendar dimension built with {len(calendar_table.columns)} columns")
print(f"   Columns: {', '.join(calendar_table.columns)}")

In [0]:
target_table = "adb_databeli.silver.calendar"

calendar_table.write.mode("overwrite").saveAsTable(target_table)

print(f"✅ Table '{target_table}' written successfully")
print(f"   📊 Total rows: {spark.table(target_table).count():,}")

display(spark.table(target_table).limit(10))